In [1]:
import preprocessamento as pp
import treinamento as tr

import pandas as pd
import yfinance as yf

In [2]:
from datetime import datetime, timedelta

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score
from sklearn.metrics import precision_recall_curve
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Parâmetros
DIAS_VALIDACAO = 90  # número de dias para simular produção
base = datetime.today().date() - timedelta(days=1)

In [5]:
df_experimentos = pd.DataFrame()

In [6]:
lista_tickers = ['ABEV3.SA', 'AZUL4.SA', 'B3SA3.SA', 'BBAS3.SA', 'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BEEF3.SA', 'BPAC11.SA', 'BRAP4.SA', 'BRDT3.SA', 'BRFS3.SA', 'BRKM5.SA', 'BRML3.SA', 'BTOW3.SA', 'CCRO3.SA', 'CIEL3.SA', 'CMIG4.SA', 'COGN3.SA', 'CPFE3.SA', 'CRFB3.SA', 'CSAN3.SA', 'CSNA3.SA', 'CVCB3.SA', 'CYRE3.SA', 'ECOR3.SA', 'EGIE3.SA', 'ELET3.SA', 'ELET6.SA', 'EMBR3.SA', 'ENBR3.SA', 'ENGI11.SA', 'EQTL3.SA', 'EZTC3.SA', 'FLRY3.SA', 'GGBR4.SA', 'GNDI3.SA', 'GOAU4.SA', 'HAPV3.SA', 'HGTX3.SA', 'HYPE3.SA', 'IGTA3.SA', 'IRBR3.SA', 'ITSA4.SA', 'ITUB4.SA', 'JBSS3.SA', 'KLBN11.SA', 'LAME4.SA', 'LREN3.SA', 'MGLU3.SA', 'MRFG3.SA', 'MRVE3.SA', 'MULT3.SA', 'NTCO3.SA', 'PCAR3.SA', 'PETR3.SA', 'PETR4.SA', 'PRIO3.SA', 'QUAL3.SA', 'RADL3.SA', 'RAIL3.SA', 'RENT3.SA', 'SANB11.SA', 'SBSP3.SA', 'SULA11.SA', 'SUZB3.SA', 'TAEE11.SA', 'TIMS3.SA', 'TOTS3.SA', 'UGPA3.SA', 'USIM5.SA', 'VALE3.SA', 'VIVT4.SA', 'VVAR3.SA', 'WEGE3.SA', 'YDUQ3.SA']
#lista_tickers = ['HAPV3.SA', 'HGTX3.SA', 'HYPE3.SA', 'IGTA3.SA', 'IRBR3.SA', 'ITSA4.SA', 'ITUB4.SA', 'JBSS3.SA', 'KLBN11.SA', 'LAME4.SA', 'LREN3.SA', 'MGLU3.SA', 'MRFG3.SA', 'MRVE3.SA', 'MULT3.SA', 'NTCO3.SA', 'PCAR3.SA', 'PETR3.SA', 'PETR4.SA', 'PRIO3.SA', 'QUAL3.SA', 'RADL3.SA', 'RAIL3.SA', 'RENT3.SA', 'SANB11.SA', 'SBSP3.SA', 'SULA11.SA', 'SUZB3.SA', 'TAEE11.SA', 'TIMS3.SA', 'TOTS3.SA', 'UGPA3.SA', 'USIM5.SA', 'VALE3.SA', 'VIVT4.SA', 'VVAR3.SA', 'WEGE3.SA', 'YDUQ3.SA']
lista_janelas = [6, 10, 15, 30, 60]

In [7]:
i = 0

for papel in lista_tickers:    
    try:
        df_base = pp.montar_dataset(papel, dias_total_historico=600, data_base=base)
    except Exception as e:
        print(f'Erro ao processar {papel}: {e}')
        continue
    
    ultimo_atr_pct = df_base['atr_pct'].iloc[-1]
    multiplo_atr = 1.5
    valorizacao_sugerida = ultimo_atr_pct * multiplo_atr
    # arredondar valorização sugerida para cima com duas casas
    valorizacao_sugerida = np.ceil(valorizacao_sugerida * 100) / 100
    stop = np.ceil(ultimo_atr_pct)  # stop loss sugerido como 1 x último ATR

    print(f'Último ATR %: {ultimo_atr_pct:.2%}')
    print(f'Valorização sugerida: {valorizacao_sugerida:.2%}') 
    print(f'Stop loss sugerido: {stop:.2%}')


    for d in range(1, 11):
        print(f'Simulando {d} dias de valorização futura...')
        dias_futuro = d
        df = pp.calcular_target_com_stop(df_base, valorizacao_alvo_pct=valorizacao_sugerida, dias_horizonte_alvo=d, stop_loss_pct=stop)
        
        if (df['Target'].value_counts(normalize=True).values[1] > 0.2):            
            print(f'Valorização futura de {dias_futuro} dias é viável.')
            break

    for dias_janela_x in lista_janelas:    
        print(f'Analisando janela de {dias_janela_x} dias...')    
        i += 1
        # Preparar X e y para treino
        X, y = pp.preparar_X_y_agregado(df, dias_janela_x, dropna=True)    
        X_train, y_train, X_validacao, y_validacao = tr.dividir_treino_validacao(X, y, DIAS_VALIDACAO, dias_futuro=dias_futuro)

        # modelo = RandomForestClassifier(n_estimators=200,
        #                                         max_depth=3,
        #                                         min_samples_leaf=50,
        #                                         max_features=0.2,
        #                                         random_state=42,
        #                                         n_jobs=-1,
        #                                         class_weight='balanced'

        #                                         )
        modelo = LogisticRegression(solver='liblinear', penalty='l2', random_state=42, class_weight='balanced')
        modelo.fit(X_train.values, y_train.values)

        roc_treino, roc_validacao, diferenca, pr_auc_treino, pr_auc_validacao, f1_treino, f1_validacao = tr.comparar_auc_conjuntos(modelo, X_train, y_train, X_validacao, y_validacao, verboso=False)

        df_experimentos.loc[i,'papel'] = papel
        df_experimentos.loc[i,'dias_futuro'] = dias_futuro
        df_experimentos.loc[i,'dias_janela_x'] =  dias_janela_x
        df_experimentos.loc[i,'ultimo_atr_pct'] =  ultimo_atr_pct 
        df_experimentos.loc[i, 'valorizacao'] =  valorizacao_sugerida
        df_experimentos.loc[i,'roc_treino'] =  roc_treino
        df_experimentos.loc[i,'roc_validacao'] =  roc_validacao
        df_experimentos.loc[i,'diferenca'] =  diferenca
        df_experimentos.loc[i,'pr_auc_treino'] =  pr_auc_treino
        df_experimentos.loc[i,'pr_auc_validacao'] =  pr_auc_validacao
        df_experimentos.loc[i,'f1_treino'] =  f1_treino
        df_experimentos.loc[i,'f1_validacao'] =  f1_validacao

[ABEV3.SA] Cache não existe ou está vazio. Download completo necessário.
[ABEV3.SA] Iniciando download de dados (lógica de atualização/complemento)...
[ABEV3.SA] Período de download yfinance: 2020-01-01 a 2025-07-14
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[ABEV3.SA] Cache principal ATUALIZADO com dados frescos: /palpiteiro/dados\ABEV3.csv
[ABEV3.SA] Retornando 612 linhas para o período solicitado.
✅ Usando cache local: /palpiteiro/dados\macro_20220121_20250710.csv
[^BVSP] Cache existente cobre de 2020-01-02 a 2025-07-10.
[^BVSP] Cache cobre período solicitado (2023-01-21 a 2025-07-09).
[^BVSP] Garantindo atualização dos últimos 5 dias do cache...
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[^BVSP] Últimos dias atualizados em memória com sucesso.
[^BVSP] Retornando 613 linhas para o perío

c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


Analisando janela de 10 dias...
INFO: 25 linha(s) removidas de X e y devido a NaNs (dropna='linhas').


c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


Analisando janela de 15 dias...
INFO: 20 linha(s) removidas de X e y devido a NaNs (dropna='linhas').


c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


... (618 outputs left out)

Analisando janela de 30 dias...
INFO: 5 linha(s) removidas de X e y devido a NaNs (dropna='linhas').


c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


Analisando janela de 60 dias...


c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\fsbertaggia\.conda\envs\p311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [ ]:
#df_experimentos.to_csv('experimentos_v2.csv', index=False)

In [4]:
df = pd.read_csv('experimentos_v2.csv')
df.head()

,papel,dias_futuro,dias_janela_x,ultimo_atr_pct,valorizacao,roc_treino,roc_validacao,diferenca,pr_auc_treino,pr_auc_validacao,f1_treino,f1_validacao
0,ABEV3.SA,4.0,6.0,0.019791,0.03,0.637019,0.439777,0.197243,0.421820,0.413704,0.291677,0.123612
1,ABEV3.SA,4.0,10.0,0.019791,0.03,0.522723,0.408518,0.114205,0.412751,0.480152,0.288502,0.288148
2,ABEV3.SA,4.0,15.0,0.019791,0.03,0.572263,0.742435,-0.170172,0.365960,0.624386,0.318050,0.429850
3,ABEV3.SA,4.0,30.0,0.019791,0.03,0.592138,0.235878,0.356260,0.387937,0.334228,0.269833,0.199938
4,ABEV3.SA,4.0,60.0,0.019791,0.03,0.658336,0.446249,0.212088,0.415611,0.459915,0.342372,0.356601


In [ ]:
#df_2 = pd.read_csv('experimentos_2.csv')

In [5]:
df_geral = pd.concat([df], ignore_index=True)
df_geral.head()

,papel,dias_futuro,dias_janela_x,ultimo_atr_pct,valorizacao,roc_treino,roc_validacao,diferenca,pr_auc_treino,pr_auc_validacao,f1_treino,f1_validacao
0,ABEV3.SA,4.0,6.0,0.019791,0.03,0.637019,0.439777,0.197243,0.421820,0.413704,0.291677,0.123612
1,ABEV3.SA,4.0,10.0,0.019791,0.03,0.522723,0.408518,0.114205,0.412751,0.480152,0.288502,0.288148
2,ABEV3.SA,4.0,15.0,0.019791,0.03,0.572263,0.742435,-0.170172,0.365960,0.624386,0.318050,0.429850
3,ABEV3.SA,4.0,30.0,0.019791,0.03,0.592138,0.235878,0.356260,0.387937,0.334228,0.269833,0.199938
4,ABEV3.SA,4.0,60.0,0.019791,0.03,0.658336,0.446249,0.212088,0.415611,0.459915,0.342372,0.356601


In [7]:
df_geral['dif_f1'] = df_geral['f1_validacao'] - df_geral['f1_treino']

In [11]:
df_geral[df_geral['dif_f1']<0.1].sort_values(by=['f1_validacao'], ascending=[False]).head(10)

,papel,dias_futuro,dias_janela_x,ultimo_atr_pct,valorizacao,roc_treino,roc_validacao,diferenca,pr_auc_treino,pr_auc_validacao,f1_treino,f1_validacao,dif_f1
163,ITSA4.SA,4.0,30.0,0.017870,0.03,0.670695,0.599028,0.071667,0.422011,0.584732,0.394375,0.477366,0.082991
83,CSAN3.SA,8.0,30.0,0.040990,0.07,0.649047,0.519109,0.129938,0.469296,0.561376,0.391274,0.462760,0.071486
80,CSAN3.SA,8.0,6.0,0.040990,0.07,0.696752,0.477393,0.219360,0.488576,0.525477,0.419898,0.446821,0.026923
289,WEGE3.SA,7.0,60.0,0.026813,0.05,0.619024,0.644886,-0.025862,0.407603,0.471731,0.393868,0.434428,0.040560
32,BEEF3.SA,4.0,15.0,0.037846,0.06,0.681190,0.760556,-0.079366,0.415242,0.729839,0.384723,0.430998,0.046276
212,PETR4.SA,3.0,15.0,0.019530,0.03,0.491795,0.582772,-0.090977,0.405946,0.395777,0.402071,0.421848,0.019776
198,MULT3.SA,4.0,30.0,0.022971,0.04,0.665363,0.550814,0.114549,0.451190,0.512980,0.332724,0.420615,0.087892
36,BPAC11.SA,3.0,10.0,0.024858,0.04,0.591748,0.672833,-0.081085,0.399926,0.456177,0.367266,0.420289,0.053023
31,BEEF3.SA,4.0,10.0,0.037846,0.06,0.586611,0.618833,-0.032222,0.392949,0.425468,0.343096,0.416715,0.073619
150,GOAU4.SA,4.0,6.0,0.024946,0.04,0.593185,0.614496,-0.021311,0.331318,0.484376,0.338087,0.411299,0.073212


In [12]:
df_geral.sort_values(by=['pr_auc_validacao'], ascending=[False]).head(30)

,papel,dias_futuro,dias_janela_x,ultimo_atr_pct,valorizacao,roc_treino,roc_validacao,diferenca,pr_auc_treino,pr_auc_validacao,f1_treino,f1_validacao
37,BEEF3.SA,8.0,15.0,0.045873,0.07,0.785121,0.710152,0.074968,0.551479,0.825363,0.445723,0.784262
14,B3SA3.SA,6.0,60.0,0.029865,0.05,0.840791,0.844461,-0.003670,0.642239,0.794473,0.518558,0.441595
205,MRFG3.SA,5.0,6.0,0.040430,0.07,0.779585,0.826902,-0.047317,0.588089,0.787441,0.504083,0.573671
147,FLRY3.SA,7.0,15.0,0.024547,0.04,0.767655,0.882989,-0.115333,0.597972,0.782418,0.532214,0.522342
264,RENT3.SA,5.0,60.0,0.032095,0.05,0.850567,0.790949,0.059618,0.587925,0.779796,0.461715,0.743000
39,BEEF3.SA,8.0,60.0,0.045873,0.07,0.833054,0.739486,0.093568,0.605884,0.772826,0.512033,0.392641
206,MRFG3.SA,5.0,10.0,0.040430,0.07,0.795152,0.788248,0.006903,0.594322,0.771853,0.548604,0.384880
310,WEGE3.SA,9.0,6.0,0.027673,0.05,0.754549,0.821674,-0.067126,0.572965,0.771779,0.469297,0.593440
36,BEEF3.SA,8.0,10.0,0.045873,0.07,0.742462,0.702689,0.039773,0.525735,0.770377,0.446930,0.749333
148,FLRY3.SA,7.0,30.0,0.024547,0.04,0.773159,0.881200,-0.108042,0.587808,0.760693,0.516747,0.494774


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.base import clone # Para criar novas instâncias do modelo

def avaliar_estrategia_com_walk_forward_no_treino(
        modelo_template, # Um modelo NÃO treinado
        X_super_treino: pd.DataFrame,
        y_super_treino: pd.Series,
        tamanho_janela_treino_wf: int = 250,
        tamanho_janela_validacao_wf: int = 30,
        passo_wf: int = 30
    ):
    """
    Avalia uma estratégia de modelagem usando validação walk-forward com retreinamento
    DENTRO do conjunto de "super treino".

    Retorna métricas médias calculadas nesse processo, para serem usadas como
    uma estimativa robusta e não enviesada do desempenho do modelo.

    Args:
        modelo_template: Uma instância NÃO TREINADA do modelo a ser avaliado.
        X_super_treino: O conjunto completo de features de treino.
        y_super_treino: Os alvos completos de treino.
        tamanho_janela_treino_wf: Número de amostras na janela de treino rolante.
        tamanho_janela_validacao_wf: Número de amostras na janela de validação interna.
        passo_wf: Número de amostras para deslizar as janelas.

    Returns:
        dict: Dicionário com as métricas médias (f1_medio, precisao_media, recall_medio).
    """
    if len(X_super_treino) < tamanho_janela_treino_wf + tamanho_janela_validacao_wf:
        print(f"AVISO: Dados de treino insuficientes ({len(X_super_treino)}) para o walk-forward com os tamanhos de janela especificados.")
        return {'f1_medio': 0, 'precisao_media': 0, 'recall_medio': 0}

    f1s, precisions, recalls = [], [], []
    num_janelas_executadas = 0

    # Usar .values aqui pode ser mais rápido para indexação em loops grandes
    X_wf_array = X_super_treino.values
    y_wf_array = y_super_treino.values
    n_total_amostras = len(X_wf_array)

    for inicio_treino_idx in range(0, n_total_amostras, passo_wf):
        fim_treino_idx = inicio_treino_idx + tamanho_janela_treino_wf
        inicio_validacao_idx = fim_treino_idx
        fim_validacao_idx = inicio_validacao_idx + tamanho_janela_validacao_wf

        if fim_validacao_idx > n_total_amostras:
            break

        num_janelas_executadas += 1
        X_treino_janela = X_wf_array[inicio_treino_idx : fim_treino_idx]
        y_treino_janela = y_wf_array[inicio_treino_idx : fim_treino_idx]
        X_validacao_janela = X_wf_array[inicio_validacao_idx : fim_validacao_idx]
        y_validacao_janela = y_wf_array[inicio_validacao_idx : fim_validacao_idx]
        
        # Retreinar o modelo em cada janela de treino
        modelo_janela = clone(modelo_template) # Cria uma nova instância limpa
        try:
            modelo_janela.fit(X_treino_janela, y_treino_janela)
        except Exception as e:
            print(f"  ERRO no fit da janela {num_janelas_executadas}: {e}. Pulando.")
            continue

        # Fazer previsões (threshold padrão de 0.5)
        y_pred_janela = modelo_janela.predict(X_validacao_janela)

        # Calcular e armazenar métricas
        f1s.append(f1_score(y_validacao_janela, y_pred_janela, zero_division=0))
        precisions.append(precision_score(y_validacao_janela, y_pred_janela, zero_division=0))
        recalls.append(recall_score(y_validacao_janela, y_pred_janela, zero_division=0))
    
    if num_janelas_executadas == 0:
        return {'f1_medio': 0, 'precisao_media': 0, 'recall_medio': 0}

    # Calcular as médias das métricas de todas as janelas
    resultado = {
        'f1_medio': np.mean(f1s),
        'precisao_media': np.mean(precisions),
        'recall_medio': np.mean(recalls)
    }

    return resultado


# --- SEU LOOP PRINCIPAL DE SCREENING, AGORA MODIFICADO ---

i = 0
df_experimentos = pd.DataFrame() # Inicializar o DataFrame de experimentos
lista_janelas = [6, 10, 15, 20, 30] # Sua lista de janelas para X
DIAS_VALIDACAO = 90 # Seu hold-out final
#lista_tickers = ["CSAN3", "PETR4", "VALE3"] # Exemplo
base = date.today() # Exemplo

for papel in lista_tickers:    
    try:
        # 1. Obter e preparar os dados e o target
        df_base = pp.montar_dataset(papel, dias_total_historico=700, data_base=base) # Um pouco mais de dados se possível
        
        ultimo_atr_pct = df_base['atr_pct'].iloc[-1]
        multiplo_atr = 1.5
        valorizacao_sugerida = np.ceil((ultimo_atr_pct * multiplo_atr) * 100) / 100
        stop = np.ceil(ultimo_atr_pct) / 100 # Stop precisa ser float, não int

        dias_futuro_final = None
        for d in range(1, 11):
            df_temp = pp.calcular_target_com_stop(df_base, valorizacao_alvo_pct=valorizacao_sugerida, dias_horizonte_alvo=d, stop_loss_pct=stop)
            # Checar se a classe positiva (1) existe e qual sua proporção
            if 1 in df_temp['Target'].value_counts(normalize=True):
                 if df_temp['Target'].value_counts(normalize=True)[1] > 0.20:
                    dias_futuro_final = d
                    print(f"Para {papel}, usando horizonte de {dias_futuro_final} dias (proporção positiva: {df_temp['Target'].value_counts(normalize=True)[1]:.2%})")
                    df = df_temp # Usar este df com o target calculado
                    break
        
        if dias_futuro_final is None:
            print(f"Não foi encontrado um horizonte de até 10 dias com >20% de alvos positivos para {papel}. Pulando papel.")
            continue
            
    except Exception as e:
        print(f'Erro na preparação inicial de dados para {papel}: {e}')
        continue
    
    for dias_janela_x in lista_janelas:    
        print(f'\nAnalisando {papel} com janela de {dias_janela_x} dias...')    
        i += 1
        
        # Preparar X e y
        X, y = pp.preparar_X_y_agregado(df, dias_janela_x, dropna=True)    
        if len(X) < 300: # Mínimo arbitrário de dados para ter treino+validação
            print(f"Dados insuficientes ({len(X)} linhas) após preparação. Pulando config.")
            continue
            
        X_train, y_train, X_validacao, y_validacao = tr.dividir_treino_validacao(X, y, DIAS_VALIDACAO, dias_futuro=dias_futuro_final)

        if len(X_train) < 280: # Mínimo para rodar o walk-forward (250 de treino + 30 de val)
            print(f"Conjunto de super treino ({len(X_train)} linhas) pequeno demais para walk-forward. Pulando config.")
            continue
            
        # Modelo template - será clonado dentro da função de avaliação
        modelo_template = LogisticRegression(solver='liblinear', penalty='l2', random_state=42, class_weight='balanced')
        
        # --- AVALIAÇÃO ROBUSTA DENTRO DO CONJUNTO DE TREINO ---
        # A "espiada" no conjunto de validação foi removida daqui.
        # Agora usamos uma métrica robusta calculada apenas no treino.
        metricas_wf_treino = avaliar_estrategia_com_walk_forward_no_treino(
            modelo_template=modelo_template,
            X_super_treino=X_train,
            y_super_treino=y_train,
            tamanho_janela_treino_wf=250, # Exemplo, ajuste se necessário
            tamanho_janela_validacao_wf=30,
            passo_wf=30
        )
        
        f1_wf_treino = metricas_wf_treino['f1_medio']
        print(f"F1-score médio robusto (Walk-Forward no Treino): {f1_wf_treino:.3f}")
        
        # --- LOG DOS EXPERIMENTOS ---
        # Agora o F1-score de validação é substituído pelo F1-score do walk-forward do treino
        # e não calculamos mais as métricas no conjunto de validação aqui no screening.
        
        experimento_atual = {
            'papel': papel,
            'dias_futuro': dias_futuro_final,
            'dias_janela_x': dias_janela_x,
            'ultimo_atr_pct': ultimo_atr_pct,
            'valorizacao': valorizacao_sugerida,
            'stop': stop,
            'f1_wf_treino': f1_wf_treino,
            'precisao_wf_treino': metricas_wf_treino['precisao_media'],
            'recall_wf_treino': metricas_wf_treino['recall_media']
        }
        df_experimentos = pd.concat([df_experimentos, pd.DataFrame([experimento_atual], index=[i])])


# --- ANÁLISE PÓS-SCREENING ---
# Agora você ordena seus resultados com base em uma métrica robusta, não enviesada.
print("\n--- Tabela Final de Experimentos ---")
df_experimentos_ordenados = df_experimentos.sort_values(by='f1_wf_treino', ascending=False)
print(df_experimentos_ordenados.to_string())

# Agora, para o seu melhor candidato da lista (ex: a primeira linha),
# você pode finalmente usar o conjunto de validação (hold-out) para
# ter uma estimativa final e imparcial do desempenho.
if not df_experimentos_ordenados.empty:
    melhor_config = df_experimentos_ordenados.iloc[0]
    print(f"\n--- Avaliação Final do Melhor Candidato no Hold-out ---")
    print(f"Configuração: Papel={melhor_config['papel']}, Janela X={melhor_config['dias_janela_x']} dias")
    
    # Recriar os dados para a melhor configuração
    df_base_final = pp.montar_dataset(melhor_config['papel'], dias_total_historico=700, data_base=base)
    df_final = pp.calcular_target_com_stop(df_base_final, valorizacao_alvo_pct=melhor_config['valorizacao'], dias_horizonte_alvo=melhor_config['dias_futuro'], stop_loss_pct=melhor_config['stop'])
    X_final, y_final = pp.preparar_X_y_agregado(df_final, int(melhor_config['dias_janela_x']), dropna=True)    
    X_train_final, y_train_final, X_validacao_final, y_validacao_final = tr.dividir_treino_validacao(X_final, y_final, DIAS_VALIDACAO, dias_futuro=melhor_config['dias_futuro'])

    # Treinar o modelo final com todo o conjunto de treino
    modelo_final = LogisticRegression(solver='liblinear', penalty='l2', random_state=42, class_weight='balanced')
    modelo_final.fit(X_train_final.values, y_train_final.values)

    # Avaliar no hold-out
    y_pred_validacao_final = modelo_final.predict(X_validacao_final.values)
    f1_holdout = f1_score(y_validacao_final, y_pred_validacao_final)
    prec_holdout = precision_score(y_validacao_final, y_pred_validacao_final)
    rec_holdout = recall_score(y_validacao_final, y_pred_validacao_final)

    print(f"Desempenho no Hold-Out: F1={f1_holdout:.3f}, Precisão={prec_holdout:.3f}, Recall={rec_holdout:.3f}")